# `aa.get_dssp()`

Run DSSP on per-entry PDB files and return `df_seq` with per-residue secondary-structure annotations. Each row is matched to `<pdb_folder>/<entry>.pdb`; the chain whose ATOM sequence best matches `df_seq[sequence]` is selected automatically, and the resulting DSSP codes are aligned back to the full sequence.

Requires `aaanalysis[pro]` (biopython) plus a `mkdssp` (or legacy `dssp`) binary on `PATH`.

In [ ]:
import aaanalysis as aa
import pandas as pd
import tempfile
from pathlib import Path
aa.options["verbose"] = False

# Minimal 4-residue ALA stub. Real workflows point `pdb_folder` at a directory
# of UniProt / AlphaFold / experimental PDBs named `<entry>.pdb`.
PDB_4ALA = (
    "HEADER    TEST\n"
    "ATOM      1  N   ALA A   1      27.340  24.430   2.614  1.00  0.00           N\n"
    "ATOM      2  CA  ALA A   1      26.266  25.413   2.842  1.00  0.00           C\n"
    "ATOM      3  C   ALA A   1      26.913  26.639   3.531  1.00  0.00           C\n"
    "ATOM      4  O   ALA A   1      27.886  26.463   4.263  1.00  0.00           O\n"
    "ATOM      5  N   ALA A   2      26.335  27.770   3.258  1.00  0.00           N\n"
    "ATOM      6  CA  ALA A   2      26.850  29.021   3.898  1.00  0.00           C\n"
    "ATOM      7  C   ALA A   2      26.100  29.253   5.202  1.00  0.00           C\n"
    "ATOM      8  O   ALA A   2      24.865  29.024   5.330  1.00  0.00           O\n"
    "ATOM      9  N   ALA A   3      26.847  29.694   6.204  1.00  0.00           N\n"
    "ATOM     10  CA  ALA A   3      26.348  29.974   7.547  1.00  0.00           C\n"
    "ATOM     11  C   ALA A   3      26.484  31.460   7.857  1.00  0.00           C\n"
    "ATOM     12  O   ALA A   3      27.521  32.026   7.512  1.00  0.00           O\n"
    "ATOM     13  N   ALA A   4      25.504  32.087   8.474  1.00  0.00           N\n"
    "ATOM     14  CA  ALA A   4      25.555  33.519   8.831  1.00  0.00           C\n"
    "ATOM     15  C   ALA A   4      26.748  33.781   9.741  1.00  0.00           C\n"
    "ATOM     16  O   ALA A   4      27.523  34.700   9.452  1.00  0.00           O\n"
    "TER      17      ALA A   4\n"
    "END\n"
)

tmp_dir = Path(tempfile.mkdtemp())
(tmp_dir / "P1.pdb").write_text(PDB_4ALA)

df_seq = pd.DataFrame({
    "entry":    ["P1"],
    "sequence": ["AAAA"],
})
df_seq

First call — anchor schema. The returned frame is a copy of `df_seq` with two extra columns: `ss` (list of per-residue codes) and `dssp_ok` (bool).

In [ ]:
df = aa.get_dssp(df_seq=df_seq, pdb_folder=str(tmp_dir), verbose=False)
aa.display_df(df=df, show_shape=True)

## `df_seq`, `pdb_folder`

`df_seq` must contain `entry` and `sequence` columns. Each `entry` value is used as the PDB-file basename (`<entry>.pdb`) under `pdb_folder`, and `sequence` is the target the DSSP output is aligned to. Entries must match `[A-Za-z0-9_.-]+` so the lookup cannot escape `pdb_folder`.

Rows whose PDB file is missing emit a `UserWarning` and surface as `ss=None`, `dssp_ok=False`; they are *not* dropped from the output.

In [ ]:
import warnings
df_with_gap = pd.DataFrame({
    "entry":    ["P1", "P_MISSING"],
    "sequence": ["AAAA", "CDEFG"],
})
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    df = aa.get_dssp(df_seq=df_with_gap, pdb_folder=str(tmp_dir), verbose=False)
aa.display_df(df=df, show_shape=True)

## `ss_mode`

Two encodings:

- `'ss3'` (default) — standard 3-state reduction (`H/E/C`). DSSP `H/G/I` collapse to `H`; `E/B` to `E`; everything else (including blanks) to `C`. Alignment gaps stay `-`.
- `'ss8'` — raw DSSP codes (`H/B/E/G/I/T/S`); DSSP's blank ("no SS") is rendered as `-` so a user never sees a literal space character.

In [ ]:
df3 = aa.get_dssp(df_seq=df_seq, pdb_folder=str(tmp_dir), ss_mode="ss3", verbose=False)
df8 = aa.get_dssp(df_seq=df_seq, pdb_folder=str(tmp_dir), ss_mode="ss8", verbose=False)
print("ss3:", df3["ss"].iloc[0])
print("ss8:", df8["ss"].iloc[0])

## `gap_handling`

DSSP only assigns SS to residues with ATOM coordinates, and `df_seq[sequence]` is often the full canonical sequence (longer than the resolved structure, with missing N/C termini or internal gaps). Two policies:

- `'pad'` (default) — align DSSP output to `df_seq[sequence]` so `len(ss) == len(sequence)`; unresolved or mismatched positions are filled with `-`.
- `'omit'` — drop those positions so the returned list contains only DSSP-assigned states. `len(ss) <= len(sequence)`.

Below: `sequence='AAAAEEEE'` but the PDB only resolves the four alanines, so the trailing glutamates become gaps under `'pad'` and are dropped under `'omit'`.

In [ ]:
df_long = pd.DataFrame({"entry": ["P1"], "sequence": ["AAAAEEEE"]})
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    df_pad = aa.get_dssp(df_seq=df_long, pdb_folder=str(tmp_dir),
                         gap_handling="pad", verbose=False)
    df_omit = aa.get_dssp(df_seq=df_long, pdb_folder=str(tmp_dir),
                          gap_handling="omit", verbose=False)
print("pad  (len =", len(df_pad["ss"].iloc[0]), "):", df_pad["ss"].iloc[0])
print("omit (len =", len(df_omit["ss"].iloc[0]), "):", df_omit["ss"].iloc[0])

## `verbose`

`verbose=True` (default) emits per-entry progress via `ut.print_out`: the matched chain id, the chain-vs-sequence identity, and the final SS length. Set `verbose=False` for quiet operation in pipelines.

In [ ]:
df = aa.get_dssp(df_seq=df_seq, pdb_folder=str(tmp_dir), verbose=True)